# Практична робота №6
## Описова статистика: міри положення і розсіювання, `describe()`, порівняння груп

**Виконав:** Панасюк Дмитро  
**Номер за списком:** 15  
**Варіант:** №5 — **Дніпро**  
**Регіон:** Схід  

**Мета:** отримати описову статистику вологості за допомогою `describe()`, перевірити формули для мір розсіювання та порівняти міста і регіони через булеву фільтрацію. Усі значення взято з навчальної таблиці умови.

## Завдання 1. `describe()` на даних свого варіанта

Значення вологості повітря (%) для Дніпра у порядку від січня до грудня:

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

months = ["Січ", "Лют", "Бер", "Кві", "Тра", "Чер", "Лип", "Сер", "Вер", "Жов", "Лис", "Гру"]
humidity = pd.Series([82, 79, 74, 67, 61, 59, 57, 59, 65, 72, 78, 83], index=months, name="Вологість, %")

display(humidity.to_frame())
print("Кількість місяців:", humidity.size)

,"Вологість, %"
Січ,82
Лют,79
Бер,74
Кві,67
Тра,61
Чер,59
Лип,57
Сер,59
Вер,65
Жов,72


Кількість місяців: 12


In [2]:
description = humidity.describe()
print("Описова статистика вологості у Дніпрі:")
display(description.to_frame(name="Значення"))

Описова статистика вологості у Дніпрі:


,Значення
count,12.000000
mean,69.666667
std,9.547330
min,57.000000
25%,60.500000
50%,69.500000
75%,78.250000
max,83.000000


**Пояснення рядків `describe()`:**

| Рядок | Зміст |
|---|---|
| `count` | Кількість спостережень (12 місяців). |
| `mean` | Середнє арифметичне вологості. |
| `std` | Вибіркове стандартне відхилення (за замовчуванням `ddof=1`). |
| `min` | Мінімальне значення. |
| `25%` | Перший квартиль Q1 (25-й процентиль). |
| `50%` | Медіана (другий квартиль). |
| `75%` | Третій квартиль Q3 (75-й процентиль). |
| `max` | Максимальне значення. |

**Для Дніпра:** середнє = **69,67%**, стандартне відхилення ≈ **9,55 процентного пункту**, мінімум = **57%**, Q1 = **60,5%**, медіана = **69,5%**, Q3 = **78,25%**, максимум = **83%**.

## Завдання 2. Ручні обчислення та перевірка

Обчислюємо розмах, вибіркову (`ddof=1`) і генеральну (`ddof=0`) дисперсії, стандартне відхилення, квартилі, міжквартильний розмах IQR та коефіцієнт варіації.

In [3]:
mean_h = humidity.mean()
minimum = humidity.min()
maximum = humidity.max()
range_h = maximum - minimum

# Дисперсія за формулами: ділимо суму квадратів відхилень на n-1 або n.
squared_deviations = ((humidity - mean_h) ** 2).sum()
n = len(humidity)
var_ddof1 = squared_deviations / (n - 1)
var_ddof0 = squared_deviations / n
std_h = np.sqrt(var_ddof1)
q1 = humidity.quantile(0.25)
q3 = humidity.quantile(0.75)
iqr_h = q3 - q1
cv_h = std_h / mean_h * 100

results = pd.Series({
    "Розмах (процентних пунктів)": range_h,
    "Вибіркова дисперсія (ddof=1)": var_ddof1,
    "Генеральна дисперсія (ddof=0)": var_ddof0,
    "Стандартне відхилення (ddof=1)": std_h,
    "Q1 (25%)": q1,
    "Q3 (75%)": q3,
    "IQR": iqr_h,
    "Коефіцієнт варіації, %": cv_h,
})
display(results.to_frame(name="Значення").round(4))
print(f"Різниця між дисперсіями: {var_ddof1 - var_ddof0:.4f}")
print(f"Вибіркова дисперсія більша за генеральну на {(var_ddof1 / var_ddof0 - 1) * 100:.2f}%")

,Значення
Розмах (процентних пунктів),26.0000
Вибіркова дисперсія (ddof=1),91.1515
Генеральна дисперсія (ddof=0),83.5556
Стандартне відхилення (ddof=1),9.5473
Q1 (25%),60.5000
Q3 (75%),78.2500
IQR,17.7500
"Коефіцієнт варіації, %",13.7043


Різниця між дисперсіями: 7.5960
Вибіркова дисперсія більша за генеральну на 9.09%


In [4]:
# Перевірка відповідності описовій статистиці та методам pandas.
assert np.isclose(q1, description["25%"])
assert np.isclose(q3, description["75%"])
assert np.isclose(std_h, description["std"])
assert np.isclose(var_ddof1, humidity.var(ddof=1))
assert np.isclose(var_ddof0, humidity.var(ddof=0))
assert np.isclose(range_h, description["max"] - description["min"])
print("Перевірка пройдена: розрахунки збігаються з describe() та pandas.")

Перевірка пройдена: розрахунки збігаються з describe() та pandas.


**Пояснення:** розмах дорівнює 83 − 57 = **26 процентних пунктів**. Вибіркова дисперсія ≈ **91,1515**, генеральна ≈ **83,5556**. Різниця виникає тому, що вибіркова дисперсія ділить суму квадратів відхилень на `n−1 = 11`, а генеральна — на `n = 12`. Вибіркова більша приблизно на **9,09%**. Значення квартилів **60,5** та **78,25** точно відповідають рядкам `25%` і `75%` в `describe()`. IQR = **17,75 процентного пункту**, CV ≈ **13,70%**.

## Завдання 3. Порівняння регіонів

Формуємо `DataFrame` **для всіх 10 міст** і розраховуємо середню річну вологість кожного міста по 12 стовпцях місяців (`axis=1`). Для порівняння обрано власний регіон **Схід** і регіон **Захід**. Використовуємо лише булеву фільтрацію, без `groupby()`.

In [5]:
cities_data = [
    ("Київ", "Центр", [84, 82, 78, 70, 65, 63, 61, 63, 68, 75, 81, 85]),
    ("Львів", "Захід", [85, 83, 80, 73, 68, 65, 63, 65, 70, 77, 82, 86]),
    ("Одеса", "Південь", [82, 80, 78, 74, 71, 69, 67, 68, 72, 76, 80, 83]),
    ("Харків", "Схід", [83, 80, 75, 68, 62, 60, 58, 60, 66, 73, 79, 84]),
    ("Дніпро", "Схід", [82, 79, 74, 67, 61, 59, 57, 59, 65, 72, 78, 83]),
    ("Чернігів", "Центр", [85, 83, 79, 71, 66, 63, 61, 63, 69, 76, 82, 86]),
    ("Івано-Франківськ", "Захід", [86, 84, 81, 75, 70, 67, 65, 67, 72, 78, 83, 87]),
    ("Полтава", "Центр", [84, 81, 76, 69, 63, 61, 59, 61, 67, 74, 80, 85]),
    ("Суми", "Схід", [85, 82, 77, 70, 64, 62, 60, 62, 68, 75, 81, 86]),
    ("Ужгород", "Захід", [84, 82, 79, 73, 68, 66, 64, 66, 71, 77, 82, 85]),
]

rows_data = [{"Місто": city, "Регіон": region, **dict(zip(months, values))}
             for city, region, values in cities_data]
df = pd.DataFrame(rows_data)
df["Середня_вологість_за_рік"] = df[months].mean(axis=1)
assert df.shape == (10, 15)
assert np.isclose(df.loc[df["Місто"] == "Дніпро", "Середня_вологість_за_рік"].iloc[0], mean_h)
display(df.round(2))

,Місто,Регіон,Січ,Лют,Бер,Кві,Тра,Чер,Лип,Сер,Вер,Жов,Лис,Гру,Середня_вологість_за_рік
0,Київ,Центр,84,82,78,70,65,63,61,63,68,75,81,85,72.92
1,Львів,Захід,85,83,80,73,68,65,63,65,70,77,82,86,74.75
2,Одеса,Південь,82,80,78,74,71,69,67,68,72,76,80,83,75.00
3,Харків,Схід,83,80,75,68,62,60,58,60,66,73,79,84,70.67
4,Дніпро,Схід,82,79,74,67,61,59,57,59,65,72,78,83,69.67
5,Чернігів,Центр,85,83,79,71,66,63,61,63,69,76,82,86,73.67
6,Івано-Франківськ,Захід,86,84,81,75,70,67,65,67,72,78,83,87,76.25
7,Полтава,Центр,84,81,76,69,63,61,59,61,67,74,80,85,71.67
8,Суми,Схід,85,82,77,70,64,62,60,62,68,75,81,86,72.67
9,Ужгород,Захід,84,82,79,73,68,66,64,66,71,77,82,85,74.75


In [6]:
# Булева фільтрація: порівнюємо середні за рік між містами двох регіонів.
east = df[df["Регіон"] == "Схід"]["Середня_вологість_за_рік"]
west = df[df["Регіон"] == "Захід"]["Середня_вологість_за_рік"]

print("Схід — середня річна вологість за містами:")
display(df[df["Регіон"] == "Схід"][["Місто", "Середня_вологість_за_рік"]].round(3))
print("Описова статистика регіону Схід:")
display(east.describe().to_frame(name="Схід"))

print("Захід — середня річна вологість за містами:")
display(df[df["Регіон"] == "Захід"][["Місто", "Середня_вологість_за_рік"]].round(3))
print("Описова статистика регіону Захід:")
display(west.describe().to_frame(name="Захід"))

Схід — середня річна вологість за містами:


,Місто,Середня_вологість_за_рік
3,Харків,70.667
4,Дніпро,69.667
8,Суми,72.667


Описова статистика регіону Схід:


,Схід
count,3.000000
mean,71.000000
std,1.527525
min,69.666667
25%,70.166667
50%,70.666667
75%,71.666667
max,72.666667


Захід — середня річна вологість за містами:


,Місто,Середня_вологість_за_рік
1,Львів,74.75
6,Івано-Франківськ,76.25
9,Ужгород,74.75


Описова статистика регіону Захід:


,Захід
count,3.000000
mean,75.250000
std,0.866025
min,74.750000
25%,74.750000
50%,74.750000
75%,75.500000
max,76.250000


In [7]:
compare_regions = pd.DataFrame({
    "Регіон": ["Схід", "Захід"],
    "Кількість міст": [len(east), len(west)],
    "Середня, %": [east.mean(), west.mean()],
    "std, п.п.": [east.std(ddof=1), west.std(ddof=1)],
    "CV, %": [east.std(ddof=1)/east.mean()*100, west.std(ddof=1)/west.mean()*100],
})
display(compare_regions.round(3))

,Регіон,Кількість міст,"Середня, %","std, п.п.","CV, %"
0,Схід,3,71.00,1.528,2.151
1,Захід,3,75.25,0.866,1.151


**Письмове порівняння:** середня річна вологість міст Сходу в середньому становить **71,00%**, Заходу — **75,25%**. Розкид **між середньорічними показниками міст** на Сході вищий: `std ≈ 1,528` проти `0,866` процентного пункту на Заході. Відносна мінливість також більша на Сході: **CV ≈ 2,151%** проти **1,151%**. Отже, за обома показниками міжміська варіативність вища на Сході. Це порівняння **не** описує розкид місячних значень у межах окремого міста.

## Завдання 4. Форма розподілу вологості Дніпра

Порівняємо середнє і медіану всіх 12 місячних значень.

In [8]:
median_h = humidity.median()
print(f"Середнє: {mean_h:.3f}%")
print(f"Медіана: {median_h:.3f}%")
print(f"Абсолютна різниця: {abs(mean_h - median_h):.3f} процентного пункту")
print(f"Асиметрія за коефіцієнтом pandas: {humidity.skew():.4f}")

Середнє: 69.667%
Медіана: 69.500%
Абсолютна різниця: 0.167 процентного пункту
Асиметрія за коефіцієнтом pandas: 0.0628


**Висновок:** середнє **69,67%**, а медіана **69,5%** — відмінність лише **0,17 процентного пункту**. За цим критерієм розподіл **близький до симетричного**, хоча ідеальна симетрія не доведена самим порівнянням двох чисел. Сильних викидів у навчальних значеннях немає, тому для опису типової вологості Дніпра **доречно використовувати середнє арифметичне**; медіана дає майже те саме і теж є коректною.

## Завдання 5. Порівняння варіативності двох міст

Порівнюємо Дніпро (**Схід**) із Львовом (**Захід**) за тими самими 12 місяцями. CV обчислюємо окремо для кожного міста: вибіркове `std / mean × 100%`.

In [9]:
lviv = pd.Series(df.loc[df["Місто"] == "Львів", months].iloc[0].to_numpy(dtype=float),
                 index=months, name="Львів")

compare_cities = pd.DataFrame({
    "Місто": ["Дніпро", "Львів"],
    "Регіон": ["Схід", "Захід"],
    "Середнє, %": [humidity.mean(), lviv.mean()],
    "std, п.п.": [humidity.std(ddof=1), lviv.std(ddof=1)],
    "CV, %": [humidity.std(ddof=1) / humidity.mean() * 100,
              lviv.std(ddof=1) / lviv.mean() * 100],
})
display(compare_cities.round(3))
assert np.isclose(humidity.std(ddof=1), std_h)

,Місто,Регіон,"Середнє, %","std, п.п.","CV, %"
0,Дніпро,Схід,69.667,9.547,13.704
1,Львів,Захід,74.750,8.444,11.296


**Письмове порівняння:** Дніпро має **CV ≈ 13,70%**, Львів — **≈ 11,30%**, отже, місячна вологість відносно власного середнього помітніше змінюється у Дніпрі. Для Дніпра `std ≈ 9,55` процентного пункту, для Львова `std ≈ 8,44`. Сирий `std` показує **абсолютний** розкид, але не враховує масштаб середніх (69,67% і 74,75%). CV нормує розкид на власне середнє, тому придатніший для порівняння відносної мінливості. Близькість середніх не скасовує користь CV; якщо середні майже однакові, різниця між висновками за `std` і CV просто буде меншою. Застосування CV тут можливе, оскільки вологість має змістовний нуль, а середні додатні.

## Додатковий приклад до контрольного питання про заповнення пропусків

Щоб показати механіку на конкретних числах, беремо **окремий ілюстративний набір**, не змінюючи дані Дніпра. Коли до вибірки додають її середнє, сума квадратів відхилень не зростає, але при розрахунку вибіркової дисперсії знаменник збільшується.

In [10]:
example_before = pd.Series([2.0, 4.0, 6.0])
example_after = pd.Series([2.0, 4.0, 6.0, example_before.mean()])
print("Початкові значення:", example_before.tolist(),
      f"| mean = {example_before.mean():.2f}, std = {example_before.std():.4f}")
print("Після заповнення середнім:", example_after.tolist(),
      f"| mean = {example_after.mean():.2f}, std = {example_after.std():.4f}")
assert example_after.std() < example_before.std()

Початкові значення: [2.0, 4.0, 6.0] | mean = 4.00, std = 2.0000
Після заповнення середнім: [2.0, 4.0, 6.0, 4.0] | mean = 4.00, std = 1.6330


## Письмові відповіді на контрольні питання

**1. Які міри розсіювання стоять за `std`, `25%/75%` і `min/max` у `.describe()`?**  
`std` — це вибіркове стандартне відхилення (стандартне значення `ddof=1` у pandas). `25%` і `75%` — перший та третій квартилі; їхня різниця дає **міжквартильний розмах** (IQR = 78,25 − 60,5 = **17,75**). `min` та `max` — крайні спостережені значення; їхня різниця дає **розмах** (83 − 57 = **26** процентних пунктів). Самі квартилі та мінімум/максимум — це не показники розсіювання, а вихідні величини для їх розрахунку.

**2. Чому заповнення пропуску середнім штучно зменшує стандартне відхилення?**  
Якщо заповнити пропуск середнім, нове значення лежатиме точно в центрі: воно додає **нуль** до суми квадратів відхилень, але збільшує кількість спостережень і знаменник дисперсії. Наприклад, у наборі `[2, 4, 6]` середнє = 4, вибіркове `std = 2`. Після додавання 4 матимемо `[2, 4, 6, 4]`: середнє досі 4, а `std = √(8/3) ≈ 1,633`. Тобто оцінка розкиду падає, хоча нової реальної інформації про мінливість не з'явилося.

**3. Чому порівнювати абсолютні `std` груп із різними середніми без CV може бути оманливо?**  
Тому що один і той самий абсолютний розкид має різну відносну величину при малому і великому середньому. Формула **CV = std / mean × 100%** дає частку коливань відносно середнього. Для порівняних регіонів Схід має `std ≈ 1,528`, `CV ≈ 2,151%`, а Захід — `std ≈ 0,866`, `CV ≈ 1,151%`: обидва показники підтверджують більшу мінливість на Сході. В іншому наборі ранжування за `std` і CV може відрізнятися.

**4. Чи симетричний розподіл Дніпра, і яка міра положення репрезентативніша?**  
Середня місячна вологість Дніпра дорівнює **69,67%**, медіана — **69,5%**, різниця лише **0,17 процентного пункту**. Отже, за близькістю цих показників розподіл є **приблизно симетричним**, без помітного зміщення середнього від медіани. **Середнє арифметичне** підходить як репрезентативна міра, оскільки використовує всі 12 значень; медіана теж добре характеризує центр і практично збігається із середнім.

### Підсумок

Для варіанта **№5 — Дніпро** розраховано `.describe()`, розмах, обидві дисперсії, стандартне відхилення, IQR та CV; перевірено відповідність формульних розрахунків результатам pandas. Побудовано повну таблицю з **10 міст**, порівняно **Схід і Захід** через булеву фільтрацію та порівняно **Дніпро і Львів** за відносною мінливістю місячної вологості. Усі обов'язкові запитання опрацьовано.